# LC 49 — Group Anagrams
**Difficulty:** Medium | **Category:** String / HashMap | **Pattern:** Canonical Key Grouping

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Two words are anagrams if and only if
their sorted character sequences are identical. Use that sorted
string as a HashMap key to bucket all anagrams together in one
pass.
</div>


## Official Problem Statement

Given an array of strings `strs`, group the **anagrams** together.
You can return the answer in **any order**.

An **anagram** is a word or phrase formed by rearranging the
letters of a different word or phrase, typically using all the
original letters exactly once.

**Constraints:**
- `1 <= strs.length <= 10^4`
- `0 <= strs[i].length <= 100`
- `strs[i]` consists of lowercase English letters.


## What This Is Actually Asking

You have a list of words.
Words that use the exact same letters (just rearranged) belong
in the same group.
Return all groups — each group is a list of those words.
The order of groups and words inside each group does not matter.


## Walk Through an Example by Hand

```
strs = ["eat", "tea", "tan", "ate", "nat", "bat"]

Word   | sorted(word) | HashMap key
-------+--------------+------------
"eat"  |  "aet"       |  "aet"
"tea"  |  "aet"       |  "aet"  (same bucket as "eat")
"tan"  |  "ant"       |  "ant"
"ate"  |  "aet"       |  "aet"  (same bucket)
"nat"  |  "ant"       |  "ant"  (same bucket as "tan")
"bat"  |  "abt"       |  "abt"

Result:
  "aet" → ["eat", "tea", "ate"]
  "ant" → ["tan", "nat"]
  "abt" → ["bat"]
```


## The Picture

HashMap after processing all words:

```
 Key (sorted) │  Value (group of words)
──────────────┼─────────────────────────
  "aet"       │  ["eat", "tea", "ate"]
  "ant"       │  ["tan", "nat"]
  "abt"       │  ["bat"]

How the key is built:

  "eat"  →  sorted → ['a','e','t'] → join → "aet"
  "tea"  →  sorted → ['a','e','t'] → join → "aet"  ✓ same!

defaultdict(list) auto-creates an empty list for new keys.
```

Every anagram of "eat" hashes to the same bucket key.


## When To Use This Pattern

- When items that "look different but are equivalent" need grouping,
  think canonical key.
- When two strings have the same characters in different order,
  think `sorted(s)` as key.
- When brute-force compares every pair O(n²), think one-pass HashMap
  O(n · k) where k is word length.
- When `defaultdict(list)` lets you skip "key exists?" checks,
  use it.
- When the problem says "group by equivalence", think canonical form
  HashMap.


## The Approach

For each word in the input list, sort its characters to produce
a canonical key — all anagrams of the same word produce the same
key.
Use a `defaultdict(list)` where each key maps to the list of
words sharing that sorted form.
Append each word to its group, then return all groups as a list
of lists.
Single pass — O(n · k log k) total where k is max word length.


In [ ]:
from typing import List                   # typed signatures
from collections import defaultdict       # auto-initialise empty list


In [ ]:
def test_harness(func):
    """Run test cases for Group Anagrams."""

    def normalize(groups):
        """Sort inner lists and outer list for order-independent compare."""
        return sorted([sorted(g) for g in groups])

    tests = [
        # (strs, expected)
        (
            ["eat", "tea", "tan", "ate", "nat", "bat"],
            [["ate", "eat", "tea"], ["nat", "tan"], ["bat"]]
        ),
        (
            [""],
            [[""]]
        ),
        (
            ["a"],
            [["a"]]
        ),
        (
            ["ab", "ba", "abc", "bca", "cab"],
            [["ab", "ba"], ["abc", "bca", "cab"]]
        ),
    ]
    passed = 0
    for strs, expected in tests:
        result = func(strs)
        if normalize(result) == normalize(expected):
            print(f"PASSED | strs={strs}")
            passed += 1
        else:
            print(
                f"FAILED | strs={strs}\n"
                f"  got      {normalize(result)}\n"
                f"  expected {normalize(expected)}"
            )
    print(f"\n{passed}/{len(tests)} tests passed.")


In [ ]:
def group_anagrams(strs: List[str]) -> List[List[str]]:
    """
    Given a list of strings, group all anagrams together.

    Approach:
      Sort each word's characters to produce a canonical key.
      Use defaultdict(list) to bucket words by that key.
      Return the dict values as a list of groups.

    Time:  O(n * k log k)  — n words, each sorted in O(k log k)
    Space: O(n * k)        — storing all words in HashMap
    """
    pass


# --- debug prints (expected in comments) ---
print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
# [["eat","tea","ate"],["tan","nat"],["bat"]]
print(group_anagrams([""]))
# [[""]]
print(group_anagrams(["a"]))
# [["a"]]
print(group_anagrams(["ab", "ba", "abc", "bca", "cab"]))
# [["ab","ba"],["abc","bca","cab"]]


In [ ]:
# Uncomment and run when solution is ready
# test_harness(group_anagrams)


## Complexity

| Approach             | Time           | Space  |
|----------------------|----------------|--------|
| Brute Force (pairs)  | O(n² · k)      | O(n)   |
| Sorted Key HashMap   | O(n · k log k) | O(n·k) |


## Real World Connection

At Citi, server monitoring across 6,000 endpoints produces
telemetry events with different field orderings depending on the
source agent — the canonical-key pattern normalises them into a
single schema bucket before routing to DynamoDB.
An AWS Lambda ETL job ingesting log records from multiple regions
can use a sorted field-fingerprint as a deduplication key, the
same way sorted characters identify anagrams.
Data engineers grouping metrics by "equivalent" alarm signatures
(same counters, different order) apply this pattern daily to
collapse redundant alerts into one actionable ticket.


> **Simplicity and clarity is Gold.** — Sean's Study Mantra
